# Capítulo 0 — El terreno: la cuenta AWS y los datos

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

| | |
|---|---|
| **Objetivo** | Dejar la cuenta AWS lista para los capítulos 1–15 (región, presupuesto, observabilidad, CDK, permisos, integración de S3 Tables), verificar el equipo de cada participante y cargar los datos del caso de uso en Iceberg |
| **Quién lo ejecuta** | **Parte A:** administrador o formador, **una vez por cuenta**, con credenciales de administración. **Partes B y C:** cada participante, en su equipo |
| **Duración orientativa** | Parte A: 25 min (más ~10 min de espera de Transaction Search) · Parte B: 20 min · Parte C: 20 min |

### Modelo de cuentas recomendado

| Modelo | Cuándo | Permisos del participante |
|---|---|---|
| **Una cuenta *sandbox* por participante** (AWS Organizations) | Formaciones (recomendado) | `AdministratorAccess` **dentro de su cuenta aislada**, con presupuesto y, a nivel de organización, una SCP que limite regiones |
| **Cuenta compartida** | Equipos pequeños o cuentas corporativas | Rol `NovaLabParticipant` (Parte A, paso 6) |

> Nunca uses una cuenta con cargas de producción: los labs crean roles IAM, Lambdas, pools de Cognito y recursos de AgentCore.

### Antes de abrir este notebook (terminal, carpeta `labs/`)

```bash
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
npm install -g @aws/agentcore          # AgentCore CLI (Node.js 20+)
jupyter lab
```

In [ ]:
import os, json
from lab_utils import *
state_show()

---
# PARTE A — Configuración de la cuenta (administrador, una vez por cuenta)

## A0. Parámetros de la formación

In [ ]:
CUENTA_ESPERADA = "<ACCOUNT_ID de la cuenta sandbox>"   # ← protección frente a ejecutar en la cuenta equivocada
REGION = "eu-west-1"                                     # región europea de la formación (doc §10.2)
MODELO_CUENTA = "sandbox"                                # "sandbox" (una cuenta por participante) o "compartida"
PRESUPUESTO_USD = 50                                     # presupuesto mensual de la cuenta para la formación
EMAIL_ALERTAS = "<formador@empresa.com>"                 # destinatario de las alertas de presupuesto
TAGS_LAB = {"Proyecto": "nova-assist-lab", "Formacion": "agentcore-oct2026"}

os.environ["AWS_REGION"] = REGION

## A1. Identidad y cuenta correcta

In [ ]:
import boto3

sts = boto3.client("sts")
yo = sts.get_caller_identity()
alias = boto3.client("iam").list_account_aliases().get("AccountAliases", [])
print("Cuenta :", yo["Account"], alias)
print("Identidad:", yo["Arn"])
assert yo["Account"] == CUENTA_ESPERADA, "⛔ No es la cuenta esperada: revisa credenciales o CUENTA_ESPERADA"
state_set(REGION=REGION, ACCOUNT_ID=yo["Account"], LAB_TAGS=TAGS_LAB)

## A2. Disponibilidad de AgentCore y Bedrock en la región

In [ ]:
def comprobar(nombre, fn):
    try:
        fn()
        print(f"✔ {nombre}")
        return True
    except Exception as e:  # noqa: BLE001
        print(f"✗ {nombre} → {type(e).__name__}: {e}")
        return False

comprobar("AgentCore (plano de control)", lambda: boto3.client("bedrock-agentcore-control", region_name=REGION).list_agent_runtimes())
comprobar("Bedrock inference profiles", lambda: boto3.client("bedrock", region_name=REGION).list_inference_profiles(typeEquals="SYSTEM_DEFINED"))
comprobar("Bedrock Guardrails", lambda: boto3.client("bedrock", region_name=REGION).list_guardrails())

> La disponibilidad varía **por capacidad**. Antes de la formación, revisa en la tabla de regiones de AgentCore que Evaluations, Policy, Managed Knowledge Base y Web Search están en la región elegida (doc §10.2).

## A3. Presupuesto y alertas de coste

In [ ]:
budgets = boto3.client("budgets", region_name="us-east-1")
NOMBRE_PRESUPUESTO = "nova-assist-lab"

def alerta(tipo, umbral):
    return {"Notification": {"NotificationType": tipo, "ComparisonOperator": "GREATER_THAN",
                             "Threshold": umbral, "ThresholdType": "PERCENTAGE"},
            "Subscribers": [{"SubscriptionType": "EMAIL", "Address": EMAIL_ALERTAS}]}

try:
    budgets.create_budget(
        AccountId=state_get("ACCOUNT_ID"),
        Budget={"BudgetName": NOMBRE_PRESUPUESTO, "BudgetType": "COST", "TimeUnit": "MONTHLY",
                "BudgetLimit": {"Amount": str(PRESUPUESTO_USD), "Unit": "USD"}},
        NotificationsWithSubscribers=[alerta("ACTUAL", 50), alerta("ACTUAL", 80), alerta("FORECASTED", 100)],
    )
    print("✔ Presupuesto creado")
except budgets.exceptions.DuplicateRecordException:
    print("= El presupuesto ya existía")
state_set(BUDGET_NAME=NOMBRE_PRESUPUESTO)

> Los recursos que crea la AgentCore CLI llevan las etiquetas `agentcore:project-name` y `agentcore:created-by`. Actívalas como **etiquetas de asignación de costes** en *Billing → Cost allocation tags* (aparecen tras crear los primeros recursos) para ver el coste por proyecto.

## A4. CloudWatch Transaction Search (requisito de Observability y Evaluations)

In [ ]:
acct = state_get("ACCOUNT_ID")
logs = boto3.client("logs", region_name=REGION)
xray = boto3.client("xray", region_name=REGION)

politica = {"Version": "2012-10-17", "Statement": [{
    "Sid": "TransactionSearchXRayAccess", "Effect": "Allow",
    "Principal": {"Service": "xray.amazonaws.com"}, "Action": "logs:PutLogEvents",
    "Resource": [f"arn:aws:logs:{REGION}:{acct}:log-group:aws/spans:*",
                 f"arn:aws:logs:{REGION}:{acct}:log-group:/aws/application-signals/data:*"],
    "Condition": {"ArnLike": {"aws:SourceArn": f"arn:aws:xray:{REGION}:{acct}:*"},
                  "StringEquals": {"aws:SourceAccount": acct}}}]}

destino = xray.get_trace_segment_destination()
if destino.get("Destination") == "CloudWatchLogs":
    print("= Transaction Search ya estaba activo:", destino)
else:
    logs.put_resource_policy(policyName="NovaLabTransactionSearch", policyDocument=json.dumps(politica))
    xray.update_trace_segment_destination(Destination="CloudWatchLogs")
    # Volumen de laboratorio: indexamos el 100 % de los spans. En producción, ajusta este porcentaje (coste).
    xray.update_indexing_rule(Name="Default", Rule={"Probabilistic": {"DesiredSamplingPercentage": 100}})
    print("✔ Transaction Search activado. Los spans tardan ~10 minutos en estar disponibles.")

## A5. *Bootstrap* de CDK

La AgentCore CLI despliega con CDK. Hacer el *bootstrap* aquí, con credenciales de administración, evita que los participantes necesiten permisos para crearlo.

In [ ]:
ACCT = state_get("ACCOUNT_ID")
etiquetas = " ".join(f"--tags {k}={v}" for k, v in TAGS_LAB.items())
!npx --yes aws-cdk@2 bootstrap aws://{ACCT}/{REGION} {etiquetas}

## A6. Rol del participante (solo cuenta compartida)

En el modelo *sandbox* el participante trabaja como administrador de su propia cuenta, así que **sáltate este paso**.

En una cuenta compartida, este rol agrupa los permisos que usan los labs. Combina políticas gestionadas por AWS con una política en línea que acota IAM, S3 y CDK por prefijos de nombre.

> Los nombres de los roles que genera CDK dependen de la versión de la CLI. Si un despliegue falla por IAM, revisa en CloudTrail qué recurso se denegó y amplía el patrón.

In [ ]:
if MODELO_CUENTA != "compartida":
    print("Modelo sandbox: no se crea el rol de participante")
else:
    iam = boto3.client("iam")
    acct = state_get("ACCOUNT_ID")
    ROL = "NovaLabParticipant"
    confianza = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow",
                 "Principal": {"AWS": f"arn:aws:iam::{acct}:root"}, "Action": "sts:AssumeRole"}]}
    try:
        iam.create_role(RoleName=ROL, AssumeRolePolicyDocument=json.dumps(confianza), MaxSessionDuration=43200,
                        Tags=[{"Key": k, "Value": v} for k, v in TAGS_LAB.items()])
    except iam.exceptions.EntityAlreadyExistsException:
        pass

    for politica in ["BedrockAgentCoreFullAccess", "AmazonBedrockFullAccess", "AWSLambda_FullAccess",
                     "AmazonCognitoPowerUser", "AWSCloudFormationFullAccess", "CloudWatchFullAccessV2",
                     "AWSXrayFullAccess", "AmazonAthenaFullAccess"]:
        iam.attach_role_policy(RoleName=ROL, PolicyArn=f"arn:aws:iam::aws:policy/{politica}")

    roles = [f"arn:aws:iam::{acct}:role/{p}" for p in ("nova-*", "*NovaAssist*", "*AgentCore*", "*agentcore*")]
    en_linea = {"Version": "2012-10-17", "Statement": [
        {"Sid": "RolesDelLab", "Effect": "Allow", "Resource": roles,
         "Action": ["iam:CreateRole", "iam:DeleteRole", "iam:GetRole", "iam:TagRole", "iam:UntagRole",
                    "iam:AttachRolePolicy", "iam:DetachRolePolicy", "iam:PutRolePolicy", "iam:DeleteRolePolicy",
                    "iam:GetRolePolicy", "iam:ListRolePolicies", "iam:ListAttachedRolePolicies",
                    "iam:UpdateAssumeRolePolicy", "iam:PassRole"]},
        {"Sid": "RolesVinculadosAServicio", "Effect": "Allow", "Action": "iam:CreateServiceLinkedRole", "Resource": "*"},
        {"Sid": "CDK", "Effect": "Allow", "Action": "sts:AssumeRole", "Resource": f"arn:aws:iam::{acct}:role/cdk-*"},
        {"Sid": "CDKBootstrapVersion", "Effect": "Allow", "Action": "ssm:GetParameter",
         "Resource": f"arn:aws:ssm:*:{acct}:parameter/cdk-bootstrap/*"},
        {"Sid": "BucketsDelLab", "Effect": "Allow", "Action": "s3:*",
         "Resource": ["arn:aws:s3:::nova-*", "arn:aws:s3:::nova-*/*"]},
        {"Sid": "ListadoS3", "Effect": "Allow", "Action": "s3:ListAllMyBuckets", "Resource": "*"},
        {"Sid": "S3TablesDelLab", "Effect": "Allow", "Action": "s3tables:*",
         "Resource": [f"arn:aws:s3tables:*:{acct}:bucket/nova-*", f"arn:aws:s3tables:*:{acct}:bucket/nova-*/*"]},
        {"Sid": "S3VectorsDelLab", "Effect": "Allow", "Action": "s3vectors:*",
         "Resource": [f"arn:aws:s3vectors:*:{acct}:bucket/nova-*", f"arn:aws:s3vectors:*:{acct}:bucket/nova-*/*"]},
        {"Sid": "ListadosDatos", "Effect": "Allow",
         "Action": ["s3tables:ListTableBuckets", "s3vectors:ListVectorBuckets", "glue:Get*", "glue:List*",
                    "lakeformation:GetDataAccess"], "Resource": "*"},
    ]}
    iam.put_role_policy(RoleName=ROL, PolicyName="NovaLabExtras", PolicyDocument=json.dumps(en_linea))
    arn = iam.get_role(RoleName=ROL)["Role"]["Arn"]
    state_set(PARTICIPANT_ROLE_ARN=arn)
    print(f"""
Concede a cada participante permiso para asumir el rol y que añada a ~/.aws/config:

[profile nova-lab]
role_arn = {arn}
source_profile = <su perfil de usuario>
region = {REGION}
""")

## A8. Integración de S3 Tables con los servicios de analítica (Athena)

Los labs guardan los datos de negocio de TiendaNova en **tablas Apache Iceberg en S3 Tables**, y la Lambda de pedidos las consulta con **Athena**. Para eso hay que crear una vez por cuenta y región el catálogo federado `s3tablescatalog` en el Glue Data Catalog, con control de acceso por IAM.

> En algunas regiones la integración con control por IAM no está disponible y requiere Lake Formation. Si esta celda falla, usa el botón **Enable integration** de la consola de S3 (*Table buckets*).

In [ ]:
glue = boto3.client("glue", region_name=REGION)
try:
    glue.get_catalog(CatalogId="s3tablescatalog")
    print("= El catálogo s3tablescatalog ya existía")
except glue.exceptions.EntityNotFoundException:
    glue.create_catalog(Name="s3tablescatalog", CatalogInput={
        "FederatedCatalog": {"Identifier": f"arn:aws:s3tables:{REGION}:{state_get('ACCOUNT_ID')}:bucket/*",
                             "ConnectionName": "aws:s3tables"},
        "CreateDatabaseDefaultPermissions": [], "CreateTableDefaultPermissions": [],
        "AllowFullTableExternalDataAccess": "True",
    })
    print("✔ Catálogo federado s3tablescatalog creado")

## A7. Resumen de la Parte A

In [ ]:
state_show()
print("\n✔ Cuenta preparada. Cada participante continúa con la Parte B en su equipo.")

---
# PARTE B — Preparación del equipo del participante

## B1. Herramientas instaladas

In [ ]:
import shutil, subprocess, sys

print("Python:", sys.version.split()[0], "(se requiere 3.10+)")
for cmd in (["node", "--version"], ["aws", "--version"], ["agentcore", "--version"]):
    if not shutil.which(cmd[0]):
        print(f"✗ {cmd[0]} no está en el PATH")
        continue
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"✔ {cmd[0]}:", (r.stdout or r.stderr).strip().splitlines()[0])

> Si `agentcore --version` da error, el antiguo `bedrock-agentcore-starter-toolkit` (pip) puede estar ocultando el comando: `pip uninstall bedrock-agentcore-starter-toolkit` y abre una terminal nueva. Ejecuta `agentcore update` antes de cada jornada.

## B2. Credenciales y región

In [ ]:
import boto3

# Cuenta compartida: usa el perfil del rol de participante. En el modelo sandbox, deja None.
PERFIL = None            # p. ej. "nova-lab"
if PERFIL:
    os.environ["AWS_PROFILE"] = PERFIL
REGION = "eu-west-1"
os.environ["AWS_REGION"] = REGION

print(boto3.client("sts").get_caller_identity()["Arn"])
state_set(REGION=REGION, ACCOUNT_ID=account_id())

> Si usas `PERFIL`, exporta también `AWS_PROFILE` en la terminal donde ejecutes `agentcore` para que la CLI use las mismas credenciales.

## B3. Comprobación de permisos (lectura)

In [ ]:
r = region()
pruebas = {
    "AgentCore": lambda: boto3.client("bedrock-agentcore-control", region_name=r).list_agent_runtimes(),
    "Bedrock": lambda: boto3.client("bedrock", region_name=r).list_inference_profiles(typeEquals="SYSTEM_DEFINED"),
    "Lambda": lambda: boto3.client("lambda", region_name=r).list_functions(MaxItems=1),
    "Cognito": lambda: boto3.client("cognito-idp", region_name=r).list_user_pools(MaxResults=1),
    "S3": lambda: boto3.client("s3").list_buckets(),
    "CDK bootstrap (CDKToolkit)": lambda: boto3.client("cloudformation", region_name=r).describe_stacks(StackName="CDKToolkit"),
    "Transaction Search": lambda: print("   destino de trazas:", boto3.client("xray", region_name=r).get_trace_segment_destination().get("Destination")),
}
for nombre, fn in pruebas.items():
    try:
        fn()
        print(f"✔ {nombre}")
    except Exception as e:  # noqa: BLE001
        print(f"✗ {nombre} → {type(e).__name__}: {e}")

Si falla **CDK bootstrap**, la Parte A no se ha ejecutado en esta región; en el modelo *sandbox*, `agentcore deploy --yes` lo hará en el primer despliegue. Si **Transaction Search** no devuelve `CloudWatchLogs`, ejecuta el paso A4.

## B4. Inference profiles EU disponibles y elección del modelo

In [ ]:
bedrock = boto3.client("bedrock", region_name=region())
perfiles, token = [], None
while True:
    kwargs = {"typeEquals": "SYSTEM_DEFINED", **({"nextToken": token} if token else {})}
    resp = bedrock.list_inference_profiles(**kwargs)
    perfiles += resp["inferenceProfileSummaries"]
    token = resp.get("nextToken")
    if not token:
        break

for pid in sorted(p["inferenceProfileId"] for p in perfiles if p["inferenceProfileId"].startswith("eu.anthropic")):
    print(pid)

In [ ]:
MODEL_ID = "eu.anthropic.<MODELO>"   # ← copia uno de la lista anterior
assert "<" not in MODEL_ID, "Elige un inference profile de la lista"
state_set(MODEL_ID=MODEL_ID)

## B5. Prueba de humo con la API Converse

In [ ]:
rt = boto3.client("bedrock-runtime", region_name=region())
resp = rt.converse(modelId=state_get("MODEL_ID"),
                   messages=[{"role": "user", "content": [{"text": "Responde solo: OK"}]}])
print(resp["output"]["message"]["content"][0]["text"])

**Si falla:**
- `ValidationException` sobre *on-demand throughput* → usas el `modelId` base en lugar del inference profile.
- `AccessDeniedException` → faltan permisos sobre el perfil y sus modelos de destino, o una SCP bloquea alguna región de destino (doc §2.2 y §2.4).
- La primera vez que se usan modelos de un proveedor en una cuenta, Bedrock puede pedir un formulario de caso de uso.

## B6. Precio del modelo para el taxímetro

Cada capítulo muestra cuántos *tokens* consume cada conversación (🚕). Si indicas el precio del modelo elegido (página de precios de Amazon Bedrock, por millón de *tokens* de entrada y de salida), el taxímetro mostrará también el coste estimado. Si lo dejas en `None`, solo verás *tokens*.

In [ ]:
PRECIO_ENTRADA_MTOK = None   # ← USD por millón de tokens de entrada (página de precios de Bedrock)
PRECIO_SALIDA_MTOK = None    # ← USD por millón de tokens de salida
if PRECIO_ENTRADA_MTOK is not None and PRECIO_SALIDA_MTOK is not None:
    precios(PRECIO_ENTRADA_MTOK, PRECIO_SALIDA_MTOK)

In [ ]:
state_show()

---
# PARTE C — Plataforma de datos del caso de uso (cada participante)

En la historia, **TiendaNova ya tiene una plataforma de datos**: sus pedidos, clientes y devoluciones viven en tablas Apache Iceberg mucho antes de que exista ningún agente. Aquí la creamos para el curso:

| Recurso | Contenido |
|---|---|
| Bucket S3 de laboratorio | Documentos fuente (`politicas/`) y resultados de Athena (`athena/`) |
| ***Table bucket* de S3 Tables** | Tablas **Apache Iceberg** del namespace `tiendanova`: `clientes`, `productos`, `pedidos`, `lineas_pedido`, `envios_eventos`, `devoluciones`, `documentos`, `eval_casos` (y `chunks`, que se llena en el capítulo 8) |

> El **índice vectorial** (S3 Vectors) no se crea aquí: aparecerá en el capítulo 8, cuando la historia lo necesite.
>
> El dataset es **sintético y reproducible**, sin datos personales reales. Sus fechas son **relativas al día de la carga**: si la formación empieza semanas después, vuelve a ejecutar esta parte para recargarlo. El laboratorio asume **un participante por cuenta y región**, porque los nombres de los recursos son fijos.

## C1. Generar el dataset

In [ ]:
import json
from dataset_tiendanova import a_json, generar

ds = generar()
DATOS_DIR.mkdir(exist_ok=True)
(DATOS_DIR / "tiendanova.json").write_text(json.dumps(ds, default=a_json, ensure_ascii=False), encoding="utf-8")
for tabla, filas in ds.items():
    print(f"{tabla:16} {len(filas):6} filas")

**Registros ancla** que usan los labs:

| Pedido | Cliente | Situación |
|---|---|---|
| TN-1001 | C-001 | Enviado, llega en 2 días |
| TN-1002 | C-002 | Entregado hace 33 días: **fuera de plazo** de devolución (y de otro cliente para C-001) |
| TN-1003 | C-001 | Entregado hace 15 días, dos artículos y cupón del 10 %: reembolso parcial (capítulo 11) |
| TN-1004 | C-001 | Auriculares in-ear entregados: excepción de higiene |
| TN-1005 | C-003 | Enviado con incidencia de retraso |

## C2. Bucket S3 del laboratorio

In [ ]:
acct, r = state_get("ACCOUNT_ID"), region()
s3 = boto3.client("s3", region_name=r)
LAB_BUCKET = f"nova-lab-{acct}-{r}"
try:
    s3.create_bucket(Bucket=LAB_BUCKET, CreateBucketConfiguration={"LocationConstraint": r})
except s3.exceptions.BucketAlreadyOwnedByYou:
    pass
for d in ds["documentos"]:
    s3.put_object(Bucket=LAB_BUCKET, Key=f"politicas/{d['doc_id']}.md", Body=d["contenido_md"].encode())
state_set(LAB_BUCKET=LAB_BUCKET, ATHENA_OUTPUT=f"s3://{LAB_BUCKET}/athena/")

## C3. *Table bucket* y namespace Iceberg

In [ ]:
s3t = boto3.client("s3tables", region_name=r)
TABLE_BUCKET = f"nova-datos-{acct}"
from botocore.exceptions import ClientError
try:
    arn = s3t.create_table_bucket(name=TABLE_BUCKET)["arn"]
except ClientError as e:
    if e.response["Error"]["Code"] != "ConflictException":
        raise
    arn = next(b["arn"] for b in s3t.list_table_buckets()["tableBuckets"] if b["name"] == TABLE_BUCKET)
state_set(TABLE_BUCKET=TABLE_BUCKET, TABLE_BUCKET_ARN=arn, ATHENA_CATALOG=f"s3tablescatalog/{TABLE_BUCKET}")

cat = catalogo_iceberg()          # endpoint Iceberg REST de S3 Tables con PyIceberg
if (ICEBERG_NAMESPACE,) not in cat.list_namespaces():
    cat.create_namespace(ICEBERG_NAMESPACE)
print("namespaces:", cat.list_namespaces())

## C4. Cargar las tablas Iceberg

In [ ]:
for tabla in ("clientes", "productos", "pedidos", "lineas_pedido", "envios_eventos", "devoluciones",
              "documentos", "eval_casos"):
    escribir_tabla(tabla, ds[tabla], modo="overwrite")

## C5. Verificar con Athena (lo mismo que hará la Lambda del capítulo 7)

In [ ]:
for fila in athena("SELECT estado, count(*) AS pedidos FROM pedidos GROUP BY estado ORDER BY pedidos DESC"):
    print(fila)
print(athena("SELECT pedido_id, cliente_id, estado, fecha_entrega FROM pedidos WHERE pedido_id = ?", ["'TN-1002'"]))

> Si Athena no ve las tablas, revisa el paso A8 (catálogo `s3tablescatalog`). Los parámetros de texto de Athena van entre comillas simples (`"'TN-1002'"`); en la Lambda se validan antes con una expresión regular, porque los escribe un modelo.

In [ ]:
state_show()


In [ ]:
guardar_partida(0)   # 💾 punto de guardado: el terreno está listo

## 🗺️ Antes de empezar la historia

Abre [**el mapa del agente**](teoria/index.html). Son las piezas que Nova Assist irá necesitando para atender a Laura; cada capítulo enciende una. Usa el interruptor para ver cuáles existen en tu máquina y cuáles solo aparecen cuando el agente sale a Internet.

> En este curso **se aprende haciendo**: empiezas en el laboratorio y la teoría llega cuando la necesitas. Cada notebook te avisará con un 🧭 en el momento justo.
>
> Las páginas de teoría son HTML interactivo. En JupyterLab se abren sin ejecutar *scripts* hasta que pulsas **Trust HTML**; también puedes abrirlas directamente en tu navegador.

---
➡️ **Siguiente capítulo:** `01_agente_simple.ipynb` — *El agente que inventa*